# Milestone 2: RAG Exploration

This notebook has the initial explorations of the RAG pipeline.

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
import faiss
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from groq import Groq
import os

In [32]:
# -------------------------
# Load data
# -------------------------
meta = pd.read_json("../data/raw/meta_Toys_and_Games.jsonl", lines=True, nrows=50000)
review = pd.read_json("../data/raw/Toys_and_Games.jsonl", lines=True, nrows=50000)

cleaned_meta = meta.drop(columns=['videos', 'price', 'images', 'bought_together', 'subtitle', 'author'], errors='ignore')
cleaned_meta.head()

reviews = review[review['verified_purchase'] == True]
cleaned_reviews = reviews.drop(columns=['images', 'timestamp', 'user_id', 'verified_purchase'], errors='ignore')
cleaned_reviews.head()

,rating,title,text,asin,parent_asin,helpful_vote
0,5,Granddaughters love them!,I purchased several of these for my granddaugh...,B09QH7QJS7,B09QH7QJS7,0
1,3,Arrived broken,"It’s cute, but it arrived broken & has some pr...",B06XYKSKQP,B06XYKSKQP,1
2,5,Dylan loves them!!!,Bough for my grandson Dylan. He loves them.,B07SFF3YQW,B07XRSD5R9,0
3,5,Janod quality and very cute...,This was great for my daughters circus themed ...,B007JWWUDW,B007JWWUDW,0
4,3,I used for my daughters circus birthday party ...,This is very cute but beware that the animals ...,B00MZG6OO8,B00MZG6OO8,2


In [33]:
# -------------------------
# Clean text columns
# -------------------------
cleaned_meta['description'] = cleaned_meta['description'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else (x if isinstance(x, str) else "")
).str.lower()

cleaned_meta['details'] = cleaned_meta['details'].apply(
    lambda x: " ".join([f"{k} {v}" for k, v in x.items()]) if isinstance(x, dict) else ""
).str.lower()

cleaned_meta['features'] = cleaned_meta['features'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
).str.lower()

cleaned_meta['categories'] = cleaned_meta['categories'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
).str.lower()

cleaned_meta['title'] = cleaned_meta['title'].str.lower()

cleaned_meta = cleaned_meta[
    (cleaned_meta['title'].str.strip() != '') &
    (cleaned_meta['description'].str.strip() != '') &
    (cleaned_meta['features'].str.strip() != '') &
    (cleaned_meta['categories'].str.strip() != '')
].reset_index(drop=True)

In [34]:
# -------------------------
# Prepare review text per product
# -------------------------
cleaned_reviews = cleaned_reviews.copy()
review_text_cols = [col for col in ['title', 'text'] if col in cleaned_reviews.columns]
cleaned_reviews['combined_review_text'] = cleaned_reviews[review_text_cols].fillna('').agg(' '.join, axis=1)
cleaned_reviews['combined_review_text'] = cleaned_reviews['combined_review_text'].str.lower()

grouped_reviews = (
    cleaned_reviews.groupby('parent_asin')['combined_review_text']
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

rag_df = cleaned_meta.merge(grouped_reviews, on='parent_asin', how='left')
rag_df['combined_review_text'] = rag_df['combined_review_text'].fillna('')

In [35]:
# -------------------------
# Build product documents
# -------------------------
products = (
    rag_df['title'] + ' ' +
    rag_df['description'] + ' ' +
    rag_df['features'] + ' ' +
    rag_df['categories'] + ' ' +
    rag_df['combined_review_text']
).tolist()

In [36]:
# -------------------------
# Embeddings and vector store
# -------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")
product_embeddings = model.encode(products).astype("float32")

index = faiss.IndexFlatL2(product_embeddings.shape[1])
index.add(product_embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7016.39it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
# -------------------------
# Retriever
# -------------------------

def retrieve(query, top_k=5):
    query_embedding = model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return rag_df.iloc[indices[0]]

# -------------------------
# Building context
# -------------------------
# Implemented with the help of chatGPT
def build_context(docs):
    blocks = []
    for _, row in docs.iterrows():
        review_snippet = row.get('combined_review_text', '')[:500]

        block = (
            f"Product ASIN: {row.get('parent_asin', 'N/A')}\n"
            f"Title: {row.get('title', '')}\n"
            f"Description: {row.get('description', '')}\n"
            f"Features: {row.get('features', '')}\n"
            f"Categories: {row.get('categories', '')}\n"
            f"Review Evidence: {review_snippet}\n"
        )
        blocks.append(block)
    return "\n\n".join(blocks)


# -------------------------
# Wrapper function
# -------------------------
def retrieve_and_build_context(query):
    docs = retrieve(query, top_k=5)
    return build_context(docs)

In [38]:
# -------------------------
# Prompt variants
# -------------------------
prompt1 = ChatPromptTemplate.from_template(

"""
You must answer using ONLY the information in the context.

- If the answer is present, extract and summarize it clearly.
- Do NOT say "I don't know" if the answer exists in the context.
- Only say "I don't know" if the context truly does not contain the answer.

Context:
{context}

Question:
{input}

Answer:
"""
)

prompt2 = ChatPromptTemplate.from_template(

"""
You must answer using ONLY the information in the context.

- Keep the answer shorter than 3 sentences.
- Make sure nothing is repeated, and only necessary details are written.
- If the answer is not in the context, say: "The context does not provide enough information."

Context:
{context}

Input:
{input}

Answer:
"""
)

prompt3 = ChatPromptTemplate.from_template(

"""
You must answer using ONLY the information in the context.

- Be clear and helpful
- Give specific statements instead of general ones. 
- If something is missing, say "not enough context to answer your question"

Context:
{context}

Question:
{input}

Answer:
"""
)

In [39]:
import os
os.environ["GROQ_API_KEY"] = "REMOVED_GROQ_API_KEY"

In [40]:
# -------------------------
# Rag Pipeline
# -------------------------
# Implemented with the help of chatGPT

llm = ChatGroq(model="llama-3.1-8b-instant")

prompts = {
    "prompt1": prompt1,
    "prompt2": prompt2,
    "prompt3": prompt3
}

query = "A good board game for kids age 8 and up"

for name, prompt in prompts.items():
    test_chain = (
        {
            "context": RunnableLambda(retrieve_and_build_context),
            "input": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    print(f"\n===== {name} =====")
    print(test_chain.invoke(query))


===== prompt1 =====
Based on the context provided, the answer is as follows:

From the description of the Sorry! board game, it's mentioned that this classic game of luck, strategy, and determination is "easy to grasp for children as young as 6 years old, yet it's fun for adults and older siblings too." However, the description also states that "this kind of frustration may be hard for children under age 8 to handle," which implies that children under 8 might not be suitable for this game.

However, the age limit provided in the product title is 6 and up, and the description does not explicitly state that it's not recommended for 8-year-olds.

===== prompt2 =====
The context does not provide enough information to determine if a specific product is suitable for kids age 8 and up.

===== prompt3 =====
Based on the information provided in the context, the Sorry! board game for kids ages 6 and up (Product ASIN: B00000IWD0) seems suitable for kids age 8 and up. It's described as a classic 

In [42]:
rag_chain = (
    {
        "context": RunnableLambda(retrieve_and_build_context),
        "input": RunnablePassthrough()
    }
    | prompt1
    | llm
    | StrOutputParser()
)

queries = [
    "A good board game for kids age 8 and up",
    "A toy for toddlers",
    "Educational toys for kids"
]

for q in queries:
    print(f"\nQUERY: {q}")
    print(rag_chain.invoke(q))


QUERY: A good board game for kids age 8 and up
The description of the Sorry! board game states that children under age 8 may have difficulty handling the frustration of the game. However, it also mentions that the game is suitable for children as young as 6 years old. Since the question asks for a good board game for kids age 8 and up, this game might be suitable, but it is recommended to consider children's emotional maturity and ability to handle frustration.

QUERY: A toy for toddlers
There are multiple toys for toddlers in the given context. Some of the relevant products are:

1. dopomt mini musical toys for toddler (Product ASIN: B09NWF6N65)
2. Disney Frozen Toddler Toy (Product ASIN: B0B625BYZM)
3. Fisher-Price Little People Monkey (Product ASIN: B07HY1XFMH)
4. Evokids Wooden Sorting Stacking Toy (Product ASIN: B09RN8RXDH)
5. Diffybox Toddler Musical Instruments (Product ASIN: B0BRZ22GNF)

All of these products are designed for toddlers and cater to their developmental needs.

Q